Part 1 - Crude Monte Carlo

In [16]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm, t

In [ ]:
n = 100

u = np.random.uniform(0, 1, n)
x = np.exp(u)
theta_hat = np.mean(x)

s = np.std(x, ddof=1)

half_width = t.ppf(0.975, n-1) * s / np.sqrt(n)

lower = theta_hat - half_width
upper = theta_hat + half_width

print("Estimate =", theta_hat)
print("95% CI =", (lower, upper))

Estimate = 1.6659956316303495
95% CI = (np.float64(1.5827091057329508), np.float64(1.7492821575277482))


In [ ]:
R = 1000
estimates = []

for _ in range(R):
    u = np.random.uniform(0, 1, n)
    estimates.append(np.mean(np.exp(u)))

print("Mean estimate =", np.mean(estimates))
print("Estimator variance =", np.var(estimates))

Mean estimate = 1.716599394786136
Estimator variance = 0.0023008648938025204


Part 2 - Antithetic Variables

In [ ]:
n = 100

u = np.random.uniform(0,1,n)
y = (np.exp(u) + np.exp(1-u)) / 2

theta_hat = np.mean(y)
s = np.std(y, ddof=1)

half_width = (t.ppf(0.975,n-1) * s / np.sqrt(n))

print(theta_hat)
print(theta_hat-half_width, theta_hat+half_width)

1.7127794132317886
1.7004375679438382 1.725121258519739


In [ ]:
R = 1000
mc = []
av = []

for _ in range(R):
    u = np.random.uniform(0,1,n)
    mc.append(np.mean(np.exp(u)))
    av.append(np.mean((np.exp(u)+np.exp(1-u))/2))

print("MC variance:",
      np.var(mc))

print("AV variance:",
      np.var(av))

MC variance: 0.002229546679900866
AV variance: 4.0025737133516306e-05


Part 3 - Control Variates

In [1]:
import numpy as np

n = 100

u = np.random.uniform(0, 1, n)

x = np.exp(u)

theta_mc = np.mean(x)

print(theta_mc)

1.7049242150975379


In [2]:
import numpy as np

n = 100

u = np.random.uniform(0, 1, n)

x = np.exp(u)

mu_z = 0.5

cov_xz = np.cov(x, u, ddof=1)[0, 1]

var_z = np.var(u, ddof=1)

c = -cov_xz / var_z

y = x + c * (u - mu_z)

theta_cv = np.mean(y)

print(theta_cv)

1.7278561692416832


In [3]:
from scipy.stats import norm
import numpy as np

alpha = 0.05

std = np.std(y, ddof=1)

half_width = (
    norm.ppf(0.975)
    * std
    / np.sqrt(n)
)

lower = theta_cv - half_width
upper = theta_cv + half_width

print("Estimate:", theta_cv)
print("95% CI:", (lower, upper))

Estimate: 1.7278561692416832
95% CI: (np.float64(1.714222173312291), np.float64(1.7414901651710752))


In [4]:
print("MC variance:", np.var(x, ddof=1))
print("CV variance:", np.var(y, ddof=1))

reduction = (
    1
    - np.var(y, ddof=1)
      / np.var(x, ddof=1)
)

print("Variance reduction:", reduction)

MC variance: 0.281927700915916
CV variance: 0.004838938894810042
Variance reduction: 0.9828362417772731


Part 7 - Importance Sampling for P(Z>a)

In [ ]:
a = 4
n = 100000

z = np.random.normal(0,1,n)

estimate = np.mean(z > a)

print(estimate)

9e-05


In [ ]:
a = 4

y = np.random.normal(a,1,n)

In [ ]:
weights = (norm.pdf(y,0,1) / norm.pdf(y,a,1))

In [ ]:
estimate = np.mean((y>a) * weights)

print(estimate)

3.1863948739686744e-05


In [ ]:
R = 500

mc = []
is_est = []

for _ in range(R):
    z = np.random.normal(0,1,n)

    mc.append(np.mean(z>a))

    y = np.random.normal(a,1,n)
    w = (norm.pdf(y,0,1)/norm.pdf(y,a,1))

    is_est.append(np.mean((y>a)*w))

print(np.var(mc))
print(np.var(is_est))

3.260144e-10
4.2643927187102915e-14


Part 8 - Importance Sampling for the Integral

In [ ]:
def sample_trunc_exp(n, lam):

    u = np.random.uniform(0, 1, n)

    return (-np.log(1-u*(1-np.exp(-lam)))/ lam)

In [ ]:
def g(x,lam):

    return (lam*np.exp(-lam*x) / (1-np.exp(-lam)))

In [ ]:
lam = 1

x = sample_trunc_exp(10000, lam)

weights = (np.exp(x) / g(x,lam))

estimate = np.mean(weights)

print(estimate)

1.7018441549946708


In [ ]:
lambdas = np.linspace(0.1, 3, 50)

variances = []

for lam in lambdas:
    x = sample_trunc_exp(10000, lam)
    w = (np.exp(x) / g(x,lam))

    variances.append(np.var(w))

best = lambdas[np.argmin(variances)]

print(best)

0.1


Part 9 - Pareto Mean via Importance Sampling

In [ ]:
def pareto_pdf(x, beta, k):
    return (k/beta * (x/beta)**(-k-1))

In [ ]:
def pareto_mean(beta, k):
    return (beta*k/(k-1))

In [ ]:
def first_moment_pdf(x, beta, k):
    mu = pareto_mean(beta, k)

    return (x * pareto_pdf(x,beta,k) / mu)

In [ ]:
x = np.linspace(1, 100, 1000)

mu = pareto_mean(1, 3)

weights = (x * pareto_pdf(x,1,3) / first_moment_pdf(x, 1, 3))

print(np.min(weights))
print(np.max(weights))

1.5
1.5
